In [26]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [27]:
df = pd.read_csv("12-house_energy_regression.csv")

In [28]:
df.head()

,avg_indoor_temp_change,outdoor_humidity_level,daily_energy_consumption_kwh
0,-0.167118,0.146714,-14.996950
1,-0.020902,0.117327,-12.678089
2,0.150419,0.364961,17.775455
3,0.555604,0.089581,6.661465
4,0.058209,-1.142970,-14.195530


In [29]:
X = df.drop("daily_energy_consumption_kwh", axis = 1)
y = df["daily_energy_consumption_kwh"]

In [30]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.25,random_state=15)

In [31]:
from sklearn.tree import DecisionTreeRegressor
tree_model = DecisionTreeRegressor()
tree_model.fit(X_train,y_train)
y_pred = tree_model.predict(X_test)

In [32]:
from sklearn.metrics import r2_score
print("r2 score: ", r2_score(y_test,y_pred))

r2 score:  0.877414941422097


In [33]:
param_grid = {
    "criterion": ["squared_error", "friedman_mse", "absolute_error"],
    "max_depth": [3, 5, 7, 10, None],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 4, 8],
    "max_features": [None, "sqrt", "log2"],
    "splitter": ["best", "random"]
}

In [34]:
from sklearn.model_selection import GridSearchCV
grid = GridSearchCV(
    estimator = DecisionTreeRegressor(),
    param_grid = param_grid,
    cv = 5,
    scoring = "r2",
    n_jobs = -1
)
grid.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=DecisionTreeRegressor(), n_jobs=-1,
             param_grid={'criterion': ['squared_error', 'friedman_mse',
                                       'absolute_error'],
                         'max_depth': [3, 5, 7, 10, None],
                         'max_features': [None, 'sqrt', 'log2'],
                         'min_samples_leaf': [1, 2, 4, 8],
                         'min_samples_split': [2, 5, 10, 20],
                         'splitter': ['best', 'random']},
             scoring='r2')

In [35]:
print("En iyi parametreler:", grid.best_params_)
print("Cross-val en iyi R2:", grid.best_score_)

En iyi parametreler: {'criterion': 'squared_error', 'max_depth': 10, 'max_features': None, 'min_samples_leaf': 4, 'min_samples_split': 5, 'splitter': 'random'}
Cross-val en iyi R2: 0.9143948473180128


In [36]:
tree_model = DecisionTreeRegressor(criterion="absolute_error", max_depth=None, max_features=None, min_samples_leaf=1,min_samples_split=20, splitter="random")
tree_model.fit(X_train,y_train)
y_pred = tree_model.predict(X_test)
print("r2 score: ", r2_score(y_test,y_pred))

r2 score:  0.9167392649208671


In [37]:
#diğer regresyon modelleriyle karşılaştırma

In [38]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import r2_score

models = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(),
    "Lasso": Lasso(),
    "Random Forest": RandomForestRegressor(),
    "Gradient Boosting": GradientBoostingRegressor(),
    "SVR": SVR(),
    "KNN": KNeighborsRegressor()
}

results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    results[name] = r2_score(y_test, y_pred)

results

{'Linear Regression': 0.9473259100085611,
 'Ridge': 0.9472814812997086,
 'Lasso': 0.9456854926210669,
 'Random Forest': 0.920429762076551,
 'Gradient Boosting': 0.9387004952847944,
 'SVR': 0.7613471586809575,
 'KNN': 0.917173862980986}